In [0]:
from pyspark.sql.functions import *

catalog = "bike_data"
silver_schema = "silver"
gold_schema = "gold"

print("=" * 80)
print("GOLD TRANSFORMATION: fact_sales")
print("=" * 80)

# Section 1: Read Silver tables
print("\nSection 1: Reading Silver tables")
sales_df = spark.table(f"{catalog}.{silver_schema}.sales")
customers_df = spark.table(f"{catalog}.{silver_schema}.customers")

print(f"  sales: {sales_df.count():,} rows")
print(f"  customers: {customers_df.count():,} rows")

# Section 2: Build fact_sales
print("\nSection 2: Building fact_sales")

# Start with sales table
fact_sales = sales_df.select(
    col("sls_ord_num"),
    col("sls_cust_id"),
    col("sls_prd_key"),
    col("sls_order_dt"),
    col("sls_ship_dt"),
    col("sls_due_dt"),
    col("sls_quantity"),
    col("sls_price"),
    col("sls_sales")
)

print(f"  Initial rows: {fact_sales.count():,}")

# Validate customer_id only (this key matches)
print("  Validating customer_id...")
fact_sales = fact_sales.join(
    customers_df.select(col("cst_id")),
    fact_sales.sls_cust_id == customers_df.cst_id,
    "inner"
)

print(f"  After customer validation: {fact_sales.count():,}")

# Select final columns
fact_sales = fact_sales.select(
    col("sls_ord_num"),
    col("sls_cust_id"),
    col("sls_prd_key"),
    col("sls_order_dt"),
    col("sls_ship_dt"),
    col("sls_due_dt"),
    col("sls_quantity"),
    col("sls_price"),
    col("sls_sales")
)

print(f"\nFinal rows in fact_sales: {fact_sales.count():,}")
print("Note: sls_prd_key kept as reference (not validated against prd_key)")

# Section 3: Sanity checks
print("\nSection 3: Sanity checks")

print("  Null values:")
null_check = fact_sales.select([count(when(col(c).isNull(), c)).alias(c) for c in fact_sales.columns])
display(null_check)

print("\n  Data sample:")
display(fact_sales.limit(3))

print("\n  Schema:")
fact_sales.printSchema()

# Section 4: Write to Gold
print("\nSection 4: Writing to Gold table")
gold_table = f"{catalog}.{gold_schema}.fact_sales"
fact_sales.write.mode("overwrite").format("delta").saveAsTable(gold_table)

final_count = fact_sales.count()
print(f"\nWritten to: {gold_table}")
print(f"Row count: {final_count:,}")